# Run CytoBridge on your data

If you are fitting a model from raw data, use the single `--train` command in
this notebook. It performs preprocessing first, then training and downstream
analysis. You do not need to run a separate preprocessing command.

Choose the included dataset that uses the most similar species, count layer,
time layout, and spatial coordinates. Export its configuration, then change the
field names and analysis settings for your AnnData object. The same edited file
is used for preprocessing, training, and downstream analysis.

If you want to see preprocessing run before using a real file, begin with the
[small generated example](data_preparation/synthetic_preprocessing.ipynb).

## Choose an example configuration

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import load_workflow_config

STARTING_CONFIG = "zebrafish"
CONFIG_PATH = Path("configs/my_dataset.json")
RAW_H5AD = Path("inputs/my_dataset_raw.h5ad")
RUN_ROOT = Path("outputs/my_dataset")
CUSTOM_LR_DATABASE = None  # or Path("inputs/my_ligand_receptor_table.csv")
RUN_WORKFLOW = False

CONFIG_TO_REVIEW = CONFIG_PATH if CONFIG_PATH.is_file() else STARTING_CONFIG
config, config_source = load_workflow_config(CONFIG_TO_REVIEW)
preprocess = config["preprocess"]
align = preprocess["align"]
dataset = config["dataset"]

spatial_obs_keys = align.get("spatial_obs_keys")
if spatial_obs_keys:
    spatial_source = ", ".join(f"obs[{key!r}]" for key in spatial_obs_keys)
else:
    spatial_source = f"obsm[{align.get('input_spatial_key', 'spatial')!r}]"

pd.DataFrame(
    {
        "field in the example": [
            "raw counts",
            "raw time",
            "raw annotation",
            "raw spatial coordinates",
            "aligned annotation",
            "aligned spatial coordinates",
        ],
        "AnnData location": [
            f"layers[{align.get('expression_layer', 'counts')!r}]",
            f"obs[{preprocess['time_key']!r}]",
            f"obs[{preprocess['annotation_source']!r}]",
            spatial_source,
            f"obs[{dataset['annotation_key']!r}]",
            f"obsm[{dataset['spatial_key']!r}]",
        ],
    }
)

,field in the example,AnnData location
0,raw counts,layers['counts']
1,raw time,obs['time']
2,raw annotation,obs['bin_annotation']
3,raw spatial coordinates,"obs['spatial_x'], obs['spatial_y']"
4,aligned annotation,obs['Annotation']
5,aligned spatial coordinates,obsm['spatial_aligned']


The table initially shows the Zebrafish example. After exporting and editing
`configs/my_dataset.json`, rerun the notebook: it will load that file and show
your field names instead.

## Check the raw AnnData layout

CytoBridge expects observations in rows and genes in columns. Before running
the workflow, check that:

- `obs_names` are unique;
- the configured count layer contains finite, non-negative integer counts;
- the configured time and annotation columns exist in `obs` and contain no
  missing values;
- every source time appears in `preprocess.align.time_mapping` and every
  observed model time appears in the data;
- the configured spatial input contains two finite coordinates for every
  observation; and
- gene names match the symbols in the ligand-receptor CSV when that analysis is
  enabled.

The [small generated example](data_preparation/synthetic_preprocessing.ipynb)
constructs an AnnData object with this layout and runs preprocessing on it.

Export the example configuration:

```bash
cytobridge workflow --config zebrafish \
  --export-config configs/my_dataset.json
```

In the exported JSON, change these exact fields:

- `dataset.name` and `dataset.annotation_key`;
- `preprocess.time_key` and `preprocess.annotation_source`;
- `preprocess.align.expression_layer`, `spatial_obs_keys` or
  `input_spatial_key`, and `time_mapping`;
- `scientific.classifier_k`, `alpha_spatial`, and `alpha_express`;
- `train.interaction_cutoff` and the training configuration when your model
  settings differ; and
- `downstream.observed`, `downstream.interpolated`, and
  `downstream.preferred_species_tag`.

The example configuration selects an LR database included with CytoBridge. To
use your own CSV instead, do not put its local path in `train.graph_database`.
Pass the file with `--graph-database` when fitting the edge predictor and with
`--lr-database` for downstream ligand--receptor analysis. The CSV must contain
`ligand` and `receptor` columns.

The five dataset notebooks display the exact raw and aligned fields used by
their included configurations.

Keep `steps.default` unchanged and `preprocess.enabled` set to `true` if you
want the single command below to run preprocessing, training, and downstream
analysis in order.

## Review the planned steps

```bash
cytobridge workflow --config configs/my_dataset.json --train \
  --input-h5ad inputs/my_dataset_raw.h5ad \
  --output-dir outputs/my_dataset --device cuda --check
```

`--check` shows the selected steps, settings, and intended paths without
starting the calculation. It does **not** open the H5AD or verify its columns,
layers, coordinates, or values. Those checks run when preprocessing starts, so
review the table above and inspect your AnnData before running the next command.

## Preprocess, train, and run downstream analysis

```bash
cytobridge workflow --config configs/my_dataset.json --train \
  --input-h5ad inputs/my_dataset_raw.h5ad \
  --output-dir outputs/my_dataset --device cuda
```

Training starts only when `--train` is present. With the exported configuration
unchanged, this command runs preprocessing, training, and downstream analysis
in order. It writes the aligned H5AD, model directory, result folders, summary
file, and PNG/PDF figures under `outputs/my_dataset`.

If you need your own LR table, use this complete version of the same command:

```bash
cytobridge workflow --config configs/my_dataset.json --train \
  --input-h5ad inputs/my_dataset_raw.h5ad \
  --graph-database inputs/my_ligand_receptor_table.csv \
  --lr-database inputs/my_ligand_receptor_table.csv \
  --output-dir outputs/my_dataset --device cuda
```

In [2]:
from CytoBridge.workflow import WorkflowOptions, run_workflow

if RUN_WORKFLOW:
    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(
            f"Export and edit the configuration before starting: {CONFIG_PATH}"
        )
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before starting: {RAW_H5AD}")
    run_config, _ = load_workflow_config(CONFIG_PATH)
    run_options = WorkflowOptions(
        input_h5ad=RAW_H5AD,
        output_dir=RUN_ROOT,
        graph_database=CUSTOM_LR_DATABASE,
        lr_database=CUSTOM_LR_DATABASE,
        device="cuda",
        train=True,
    )
    run_result = run_workflow(run_config, options=run_options)
    run_result
else:
    print(
        "The full run is off. Update the paths, then set RUN_WORKFLOW = True "
        "to preprocess, train, and run downstream analysis."
    )

The full run is off. Update the paths, then set RUN_WORKFLOW = True to preprocess, train, and run downstream analysis.


## Continue from an existing model

Use the aligned H5AD and model directory from the same run:

```bash
cytobridge workflow --config configs/my_dataset.json --step downstream \
  --aligned-h5ad outputs/my_dataset/preprocess/my_dataset_aligned.h5ad \
  --model-dir outputs/my_dataset/training \
  --output-dir outputs/my_dataset_downstream_rerun --device cuda
```

Use a new output directory for a second downstream calculation. The
paper-figure commands state whether they read this new run or files retained
from the paper analysis. A paper redraw command does not automatically analyze
this new directory.

## Expected output locations

In [3]:
with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {
                "output": [
                    "aligned data",
                    "model directory",
                    "downstream summary",
                    "standard figures",
                ],
                "path": [
                    RUN_ROOT / "preprocess" / "my_dataset_aligned.h5ad",
                    RUN_ROOT / "training",
                    RUN_ROOT / "downstream" / "summary.json",
                    RUN_ROOT / "downstream" / "figures",
                ],
            }
        )
    )

,output,path
0,aligned data,outputs/my_dataset/preprocess/my_dataset_aligned.h5ad
1,model directory,outputs/my_dataset/training
2,downstream summary,outputs/my_dataset/downstream/summary.json
3,standard figures,outputs/my_dataset/downstream/figures
